In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit

# 1. Load your real Serbia dataset
file_path = covid_file
df = pd.read_csv(file_path)
days_data = df['day'].values
cases_data = df['infected'].values

N = 6834326

# Standard RK4 Solver for a single segment
def solve_sir_rk4(beta, gamma, S0, I0, R0, days, dt):
    num_steps = int(days / dt) + 1
    S, I, R = np.zeros(num_steps), np.zeros(num_steps), np.zeros(num_steps)
    t = np.arange(num_steps) * dt
    S[0], I[0], R[0] = S0, I0, R0
    N_total = S0 + I0 + R0 
    
    for n in range(num_steps - 1):
        dS = - (beta * S[n] * I[n]) / N_total
        dI = (beta * S[n] * I[n]) / N_total - (gamma * I[n])
        dR = gamma * I[n]
        S[n+1] = S[n] + dt * dS
        I[n+1] = I[n] + dt * dI
        R[n+1] = R[n] + dt * dR
    return t, S, I, R

def sir_fit_func(t, beta, gamma):
    sim_days = np.max(t) - np.min(t)
    # Use the starting conditions of the specific chunk
    t_sim, _, I_sim, _ = solve_sir_rk4(beta, gamma, chunk_S0, chunk_I0, 0, sim_days, dt=1.0)
    return np.interp(t, np.linspace(np.min(t), np.max(t), len(I_sim)), I_sim)

# 2. Define the Three Periods
periods = [
    {"name": "Pre-Lockdown (Wave 1)", "range": (0, 125), "color": "#1f77b4"},
    {"name": "Lockdown / Lull Period", "range": (125, 225), "color": "#ff7f0e"},
    {"name": "Post-Lockdown (Autumn Wave)", "range": (225, 360), "color": "#2ca02c"}
]

# 3. Plotting Setup
plt.figure(figsize=(13, 6))
plt.scatter(days_data, cases_data, color='gray', s=12, alpha=0.5, label='Observed Daily Cases')

# 4. Fit and Plot each period independently
for p in periods:
    start, end = p["range"]
    mask = (days_data >= start) & (days_data < end)
    chunk_days = days_data[mask]
    chunk_cases = cases_data[mask]
    
    if len(chunk_cases) == 0:
        continue
        
    # Set initial conditions for this specific chunk
    chunk_I0 = chunk_cases[0]
    chunk_S0 = N - chunk_I0
    
    # Fit parameters for this chunk
    try:
        popt, _ = curve_fit(sir_fit_func, chunk_days, chunk_cases, p0=[0.3, 0.2], bounds=(0.01, 3.0))
        beta_fit, gamma_fit = popt
        
        # Simulate the fitted curve for this chunk's duration
        duration = chunk_days[-1] - chunk_days[0]
        t_sim, _, I_sim, _ = solve_sir_rk4(beta_fit, gamma_fit, chunk_S0, chunk_I0, 0, duration, dt=1.0)
        t_plot = np.linspace(chunk_days[0], chunk_days[-1], len(I_sim))
        
        # Plot chunk fit
        plt.plot(t_plot, I_sim, color=p["color"], linewidth=3, 
                 label=rf'{p["name"]} ($\beta$={beta_fit:.3f}, $\gamma$={gamma_fit:.3f})')
    except Exception as e:
        print(f"Fitting failed for {p['name']}: {e}")

# Formatting the final comprehensive plot
plt.axvspan(125, 225, color='orange', alpha=0.1, label='Lockdown Window (Days 125–225)')
plt.title('Piecewise SIR Fitting Across Politically/Socially Partitioned Periods (Serbia 2021)', fontsize=13, fontweight='bold')
plt.xlabel('Day of Year (2021)', fontsize=12)
plt.ylabel('Daily New Cases', fontsize=12)
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

NameError: name 'covid_file' is not defined